# Intent-Conditioned CFM: Evaluation Results

This notebook loads the evaluation results from `eval_intent_cfm.py`
and produces PIT histograms, error summaries, and calibration analysis.

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd()))
from intent import intent_name

In [3]:
# Load results
data = np.load("eval_intent_cfm_2000x64x64_91330.npz")
pits = data["pits"]                # (N, T, 3)
ade_mean = data["ade_mean"]        # (N,)
fde_mean = data["fde_mean"]        # (N,)
ade_single = data["ade_single"]    # (N,)
spread = data["spread"]            # (N, T)
intent_labels = data["intent_labels"]  # (N,)

print(f"Loaded {pits.shape[0]} evaluation samples, {pits.shape[1]} timesteps")
print(f"ADE (mean ensemble): {ade_mean.mean():.1f} ± {ade_mean.std():.1f} m")
print(f"ADE (single sample): {ade_single.mean():.1f} ± {ade_single.std():.1f} m")
print(f"FDE (mean ensemble): {fde_mean.mean():.1f} ± {fde_mean.std():.1f} m")
print(f"Mean spread: {spread.mean():.1f} m")

Loaded 2000 evaluation samples, 12 timesteps
ADE (mean ensemble): 104.2 ± 196.1 m
ADE (single sample): 145.3 ± 231.9 m
FDE (mean ensemble): 273.7 ± 548.5 m
Mean spread: 136.0 m


## PIT Histograms (all data)

A well-calibrated model produces PIT values that are uniform on [0,1].
- **U-shaped**: under-dispersed (ensemble too narrow)
- **Hump-shaped** (peaked at 0.5): over-dispersed (ensemble too wide)
- **Flat**: perfect calibration

In [4]:
axis_names = ["x", "y", "z"]
axis_colors = ["black", "red", "blue"]
nbins = 20

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.07,
                    subplot_titles=[f"PIT — {a}" for a in axis_names])

pits_flat = pits.reshape(-1, 3)
for d in range(3):
    u = pits_flat[:, d]
    fig.add_trace(go.Histogram(
        x=u, nbinsx=nbins, histnorm="probability density",
        marker=dict(color=axis_colors[d]), opacity=0.75,
        name=axis_names[d], showlegend=False,
    ), row=1, col=d+1)
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[1, 1], mode="lines",
        line=dict(dash="dash", color="gray", width=2),
        showlegend=(d==0), name="Uniform (ideal)",
    ), row=1, col=d+1)
    fig.update_xaxes(title_text="PIT value", range=[0, 1], row=1, col=d+1)
    fig.update_yaxes(title_text="Density", range=[0, 3], row=1, col=d+1)

fig.update_layout(
    height=350, width=900, template="plotly_white",
    title="PIT Histograms — Intent-Conditioned CFM (all horizons)",
    bargap=0.05,
)
fig.show()

## PIT by horizon

Short-horizon predictions should be well-calibrated;
long-horizon ones are harder.

In [5]:
T = pits.shape[1]
horizons = [(0, "t=+5s"), (T//4, f"t=+{(T//4+1)*5}s"), (T//2, f"t=+{(T//2+1)*5}s"), (T-1, f"t=+{T*5}s")]

fig = make_subplots(rows=len(horizons), cols=3, horizontal_spacing=0.07, vertical_spacing=0.08,
                    row_titles=[h[1] for h in horizons],
                    column_titles=axis_names)

for ri, (tidx, tname) in enumerate(horizons):
    for d in range(3):
        u = pits[:, tidx, d]
        fig.add_trace(go.Histogram(
            x=u, nbinsx=15, histnorm="probability density",
            marker=dict(color=axis_colors[d]), opacity=0.7, showlegend=False,
        ), row=ri+1, col=d+1)
        fig.add_trace(go.Scatter(
            x=[0,1], y=[1,1], mode="lines",
            line=dict(dash="dash", color="gray"), showlegend=False,
        ), row=ri+1, col=d+1)
        fig.update_xaxes(range=[0,1], row=ri+1, col=d+1)
        fig.update_yaxes(range=[0,4], row=ri+1, col=d+1)

fig.update_layout(height=200*len(horizons), width=900, template="plotly_white",
                  title="PIT by forecast horizon")
fig.show()

## PIT by intent category

The key question: does calibration differ by intent?
Level/Straight should be tighter. Turning should be wider.

In [6]:
from collections import Counter

# Group by main vertical/lateral category
v_phase = intent_labels // 5
l_phase = intent_labels % 5

# Lateral grouping (most interesting for xy calibration)
lat_names = {0: "Straight", 1: "Turn Left", 2: "Turn Right", 3: "Roll-out(L)", 4: "Roll-out(R)"}
lat_groups = {}
for lp in range(5):
    mask = l_phase == lp
    if mask.sum() >= 10:
        lat_groups[lat_names[lp]] = mask

fig = make_subplots(rows=len(lat_groups), cols=2, horizontal_spacing=0.08, vertical_spacing=0.08,
                    row_titles=list(lat_groups.keys()),
                    column_titles=["PIT x", "PIT y"])

for ri, (name, mask) in enumerate(lat_groups.items()):
    for d in range(2):
        u = pits[mask, :, d].ravel()
        fig.add_trace(go.Histogram(
            x=u, nbinsx=15, histnorm="probability density",
            marker=dict(color=axis_colors[d]), opacity=0.7, showlegend=False,
        ), row=ri+1, col=d+1)
        fig.add_trace(go.Scatter(
            x=[0,1], y=[1,1], mode="lines",
            line=dict(dash="dash", color="gray"), showlegend=False,
        ), row=ri+1, col=d+1)
        fig.update_xaxes(range=[0,1], row=ri+1, col=d+1)
        fig.update_yaxes(range=[0,4], row=ri+1, col=d+1)

fig.update_layout(height=180*len(lat_groups), width=700, template="plotly_white",
                  title="PIT (x, y) by lateral intent — does calibration differ?")
fig.show()

## Spread vs error by intent

In [7]:
# Summary table
rows = []
unique_intents = np.unique(intent_labels)
for uid in unique_intents:
    mask = intent_labels == uid
    n = int(mask.sum())
    if n < 5:
        continue
    rows.append({
        "intent": intent_name(int(uid)),
        "n": n,
        "ADE_mean_m": float(ade_mean[mask].mean()),
        "FDE_mean_m": float(fde_mean[mask].mean()),
        "spread_m": float(spread[mask].mean()),
        "PIT_x_mean": float(pits[mask, :, 0].mean()),
        "PIT_y_mean": float(pits[mask, :, 1].mean()),
        "PIT_z_mean": float(pits[mask, :, 2].mean()),
    })

import pandas as pd
df = pd.DataFrame(rows).sort_values("n", ascending=False)
print(df.to_string(index=False, float_format="%.2f"))

                   intent    n  ADE_mean_m  FDE_mean_m  spread_m  PIT_x_mean  PIT_y_mean  PIT_z_mean
         Level / Straight 1084      109.34      287.01    137.62        0.52        0.50        0.46
    Descending / Straight  393      110.02      286.74    139.31        0.50        0.49        0.48
      Climbing / Straight  357       82.71      225.10    127.82        0.51        0.48        0.46
     Climbing / Turn Left   35       90.40      231.27    147.83        0.50        0.54        0.51
  Descending / Turn Right   27      106.04      265.45    121.51        0.54        0.44        0.40
       Level / Turn Right   24       77.24      215.80    142.20        0.51        0.48        0.39
   Descending / Turn Left   22       41.66      103.34     95.86        0.46        0.47        0.48
    Climbing / Turn Right   21      152.49      350.40    139.81        0.47        0.56        0.45
        Level / Turn Left   16       83.11      233.56    135.67        0.49        0.55   

## Error vs horizon

In [8]:
# Spread over horizon
horizons_s = np.arange(1, spread.shape[1]+1) * 5
mean_spread = spread.mean(axis=0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=horizons_s, y=mean_spread, mode="lines+markers",
                         name="Mean spread", line=dict(width=2)))
fig.update_layout(title="Ensemble spread vs forecast horizon",
                  xaxis_title="Horizon (s)", yaxis_title="Spread (m)",
                  height=350, width=600, template="plotly_white")
fig.show()

## Paper-style MAE/RMSE (Model mean vs Best-of-S vs CV)

This cell matches Ben's plot style and includes all 3 curves.

Use an NPZ generated with the **updated** `eval_intent_cfm.py`.

In [ ]:

keys = set(data.files)
req = {
    "mae3d_mean_curve", "mae_xy_mean_curve", "mae_z_mean_curve",
    "rmse3d_mean_curve", "rmse_xy_mean_curve", "rmse_z_mean_curve",
    "mae3d_best_curve", "mae_xy_best_curve", "mae_z_best_curve",
    "rmse3d_best_curve", "rmse_xy_best_curve", "rmse_z_best_curve",
    "mae3d_cv_curve", "mae_xy_cv_curve", "mae_z_cv_curve",
    "rmse3d_cv_curve", "rmse_xy_cv_curve", "rmse_z_cv_curve",
}

if not req.issubset(keys):
    print("Missing paper-style keys in NPZ:")
    print(sorted(req - keys))
    print("Rerun eval_intent_cfm.py with the latest code.")
else:
    horizons_s = np.arange(1, len(data["mae3d_mean_curve"]) + 1) * 5
    C_MEAN, C_BEST, C_CV = "royalblue", "seagreen", "firebrick"

    fig = make_subplots(
        rows=3,
        cols=2,
        subplot_titles=[
            "MAE - 3D (x,y,z)", "RMSE - 3D (x,y,z)",
            "MAE - Horizontal (x,y)", "RMSE - Horizontal (x,y)",
            "MAE - Vertical (z)", "RMSE - Vertical (z)",
        ],
    )

    def add_triplet(row, col, y_mean, y_best, y_cv, showlegend=False):
        fig.add_trace(go.Scatter(
            x=horizons_s, y=y_mean, mode="lines+markers",
            name="Model mean", line=dict(color=C_MEAN, width=2), showlegend=showlegend,
        ), row=row, col=col)
        fig.add_trace(go.Scatter(
            x=horizons_s, y=y_best, mode="lines+markers",
            name="Best-of-S", line=dict(color=C_BEST, width=2), showlegend=showlegend,
        ), row=row, col=col)
        fig.add_trace(go.Scatter(
            x=horizons_s, y=y_cv, mode="lines+markers",
            name="CV baseline", line=dict(color=C_CV, width=2), showlegend=showlegend,
        ), row=row, col=col)

    add_triplet(1, 1, data["mae3d_mean_curve"], data["mae3d_best_curve"], data["mae3d_cv_curve"], showlegend=True)
    add_triplet(1, 2, data["rmse3d_mean_curve"], data["rmse3d_best_curve"], data["rmse3d_cv_curve"], showlegend=False)
    add_triplet(2, 1, data["mae_xy_mean_curve"], data["mae_xy_best_curve"], data["mae_xy_cv_curve"], showlegend=False)
    add_triplet(2, 2, data["rmse_xy_mean_curve"], data["rmse_xy_best_curve"], data["rmse_xy_cv_curve"], showlegend=False)
    add_triplet(3, 1, data["mae_z_mean_curve"], data["mae_z_best_curve"], data["mae_z_cv_curve"], showlegend=False)
    add_triplet(3, 2, data["rmse_z_mean_curve"], data["rmse_z_best_curve"], data["rmse_z_cv_curve"], showlegend=False)

    fig.update_xaxes(title_text="Horizon (s)", row=3, col=1)
    fig.update_xaxes(title_text="Horizon (s)", row=3, col=2)
    fig.update_yaxes(title_text="Meters", row=1, col=1)
    fig.update_yaxes(title_text="Meters", row=2, col=1)
    fig.update_yaxes(title_text="Meters", row=3, col=1)

    fig.update_layout(
        template="plotly_white",
        height=1050,
        width=1200,
        title="",
        legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0.0),
    )
    fig.show()